# Aprendizado por Reforço

---

**Professor:** Prof. Gabriel Lima  
**Aula:** 01  
**Exercício:** 1A  

---

### Objetivo :  
Entender o conceito de ambiente, política, agente e episódio.


## Bibliotecas Utilizadas

Nesse primeiro momento, a contrução do ambiente, assim como a definição do agente será realizada na forma elementar, sem a ajuda de frameworks. 

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import animation
from PIL import Image
import io

## Variáveis e configurações

Será utilizado um seed para a biblioteca random a fim de se obter uma reprodutibilidade no código aqui descrito.

In [ ]:
random.seed(2)

## Definição do Ambiente

**Regras:**  

O jogo inicia com o jogador na posição [0,0], onde o objetivo é chegar ao ponto [3,3].  

Buracos de gelo são distribuidos pelo grid. 

O jogador deve-se mover até atingir um estado terminal ( cair em um buraco ou chegar em seu objetivo).

Nesse exercício o ambiente será modelado como deterministico, ou seja, uma vez que o agente tomou uma decisão ela será executada.  

**Recompensas:**

* Atingir o objetivo: +1
* Cair em um buraco : 0
* Demais estados    : 0


**Estados:**

|     |     |     |     |
|-----|-----|-----|-----|
|  0  |  1  |  2  |  3  |
|  4  |  5  |  6  |  7  |
|  8  |  9  | 10  | 11  |
| 12  | 13  | 14  | 15  |

In [ ]:
class FrozenLake:
    # Definicao do Metodo Construtor
    def __init__(self):
        self.size = 4

        #  0 (Estados Terminais)
        #  1 (Estados transitórios)
        # -1 (Estado Terminal - Objetivo)

        self.grid = np.array([[1,1,1,1],
                              [1,0,1,0],
                              [1,1,1,0],
                              [0,1,1,2]])
        
        self.terminal = False
        self.Frames   = []
        self.positions= []
        
        # [1,1,1,1]
        # [1,0,1,0]
        # [1,1,1,0]
        # [0,1,1,-1] 
        # y linhas e x colunas

        # (y,x)
        self.state = (0,0)

    # Definicao do metodo de reset para o estado
    def initial_state(self):
        self.state = (0,0)
        self.terminal = False
        self.Frames = []
        self.positions = [self.state]
        return self.state
    
    def step(self,action):

        y,x = self.state

        if action == 'UP' and y > 0 and not self.terminal:
            y -=1
        
        elif action == 'DOWN' and y < self.size-1 and not self.terminal:
            y +=1

        elif action == 'LEFT' and x > 0 and not self.terminal:
            x -=1
        
        elif action == 'RIGHT' and x < self.size-1 and not self.terminal:
            x +=1

        if self.grid[y][x] == 0 or self.grid[y][x] == 2:
                self.terminal = True

        self.state = (y,x) 
        self.positions.append(self.state)   

        return self.state, 1 if (y,x) == (3,3) else 0,self.terminal
    
    def show(self):

        for num,position in enumerate(self.positions):

            fig, ax = plt.subplots()
            cax = ax.imshow(self.grid, cmap='Blues', interpolation='nearest')
            ax.grid(True, which='both', color='black', linestyle='--', linewidth=0.5)
            ax.scatter(position[1], position[0], s=128, color='r')

            ax.set_title('Step: {}'.format(num), fontsize=12, pad=20,loc='left')


            ax.set_xticks(np.arange(-0.5, 4, 1))
            ax.set_yticks(np.arange(-0.5, 4, 1))
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.set_aspect('equal', adjustable='box')

            buf = io.BytesIO()
            fig.savefig(buf, orientation='landscape')
            buf.seek(0)
            img = Image.open(buf)
            
            self.Frames.append(img)

        return self.Frames

## Definição do Agente

**Regras:**  

O agente seguirá uma política estocástica, onde: 

$\pi: \mathcal{S} \times \mathcal{A} \rightarrow [0,1]$, tal que $\pi(a \mid s) = \mathbb{P}(A_t = a \mid S_t = s)$  

Para esse exercício a política terá uma probabilidade uniforme - ou seja, a chance do agente executar uma determinada ação será igual para todas as direções.



In [ ]:
class Agent:

    def __init__(self):
        self.action_space  = ['UP','DOWN','LEFT','RIGHT']

    def next_action(self):
        action =  random.choice(self.action_space)
        return action

## Definição do episódio

Forçaremos o problema ser um episódic task, admitindo um timestep maximo de 200 passos. 

In [ ]:
def episode(environment,agent,timestep=200):
    state      = environment.initial_state()

    for _ in range(timestep):
        action = agent.next_action()
        print(f'Estado: {state} , Acao : {action}')
        state,_,terminated = environment.step(action)
        if terminated:
            print(f'Estado: {state} -> Acao : -----')
            return environment.show()

## Exportando o ambiente como um gif

Utilizaremos todos os frames salvos para a confecção de um gif.

In [ ]:
def save_frames_as_gif(frames, path='./', filename='Aula1A.gif'):

    fig, ax = plt.subplots()
    plt.axis('off')
    im = ax.imshow(frames[0], animated=True)
    
    def update(i):
        im.set_array(frames[i])
        return im, 

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=200,
                                    blit=True,repeat_delay=10,)   
    anim.save(filename)

## Instanciando os Objetos

Por fim, usaremos os objetos e funções previamente criados para obter as saídas desejadas.

In [ ]:
env        = FrozenLake()
agent      = Agent()

In [ ]:
plt.ioff()
Frames = episode(env,agent)

In [ ]:
save_frames_as_gif(Frames)